# Week 4 Hands-On Lab — Search Under Real Constraints

**ESP3201 · formative hands-on lab.** Runs on free-tier Google Colab with a **CPU runtime** (no GPU needed).

You will run a panel of classic graph-search planners on grid maps and weigh them not by path length alone, but by **search effort (nodes expanded), memory (peak frontier), runtime, and replanning cost** — the realities that decide which planner ships on a robot.

> **Report only what your run actually produced.** Every number in your worksheet must come from a cell you ran.

**Attribution.** Problem framing follows the grid-search pedagogy of Berkeley CS188 (Pacman search projects) and Russell & Norvig, *AIMA*. All lab code is original to this course.

## Tasks

1. Run the planner panel on the `maze` map and read the instrumentation.
2. Compare the panel across `open`, `maze`, and `terrain` maps.
3. Isolate the **heuristic effect**: admissible A* vs inflated heuristic vs greedy.
4. **Sweep the Weighted-A* weight yourself** and watch search effort collapse.
5. Run the **replanning** scenario.
6. Fill the worksheet.

## Setup

In [ ]:
# --- Week 4 lab core, embedded directly (no repo clone) -----------------------
# Canonical source, kept in sync by hand: starter/search_lab.py in the course
# repo. Cloning a support module from Colab is fragile -- if a session already
# ran once before an update landed, the clone silently no-ops onto the stale
# cached copy instead of fetching the fix. Pasting the needed code directly
# here removes that whole failure class.

from __future__ import annotations

import heapq
import time
from collections import deque
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple

Cell = Tuple[int, int]  # (row, col)


# --------------------------------------------------------------------------- #
# Grid world
# --------------------------------------------------------------------------- #
class Grid:
    """A 2D grid with blocked cells and optional per-cell movement cost.

    A map is given as a list of equal-length strings:
        '.' free cell (cost 1)
        '#' blocked cell
        digit '2'..'9' free cell whose *entry* cost is that digit (terrain)
        'S' start, 'G' goal
    """

    def __init__(self, rows: List[str], diagonal: bool = False):
        self.rows = rows
        self.height = len(rows)
        self.width = len(rows[0]) if rows else 0
        self.diagonal = diagonal
        self.start: Optional[Cell] = None
        self.goal: Optional[Cell] = None
        self.blocked = set()
        self.cost: Dict[Cell, int] = {}
        for r, line in enumerate(rows):
            for c, ch in enumerate(line):
                cell = (r, c)
                if ch == "#":
                    self.blocked.add(cell)
                elif ch == "S":
                    self.start = cell
                elif ch == "G":
                    self.goal = cell
                elif ch.isdigit():
                    self.cost[cell] = int(ch)

    def in_bounds(self, cell: Cell) -> bool:
        r, c = cell
        return 0 <= r < self.height and 0 <= c < self.width

    def passable(self, cell: Cell) -> bool:
        return cell not in self.blocked

    def entry_cost(self, cell: Cell) -> int:
        """Cost to step *into* `cell` (terrain cost, default 1)."""
        return self.cost.get(cell, 1)

    def neighbors(self, cell: Cell) -> List[Cell]:
        r, c = cell
        steps = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        if self.diagonal:
            steps += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
        result = []
        for dr, dc in steps:
            nxt = (r + dr, c + dc)
            if self.in_bounds(nxt) and self.passable(nxt):
                result.append(nxt)
        return result

    def block(self, cell: Cell) -> None:
        """Drop a new obstacle (used by the replanning scenario)."""
        self.blocked.add(cell)


# --------------------------------------------------------------------------- #
# Result container
# --------------------------------------------------------------------------- #
@dataclass
class SearchResult:
    algorithm: str
    found: bool
    path: List[Cell] = field(default_factory=list)
    cost: float = float("inf")
    nodes_expanded: int = 0
    max_frontier: int = 0
    runtime_s: float = 0.0
    # cells in the order popped off the frontier -- lets a plot show *how* the
    # search grew (BFS's ripple vs DFS's snake vs A*'s cone), not just the count
    expanded_order: List[Cell] = field(default_factory=list)

    @property
    def path_len(self) -> int:
        return len(self.path)

    def as_row(self) -> Dict[str, object]:
        return {
            "algorithm": self.algorithm,
            "found": self.found,
            "cost": round(self.cost, 2) if self.found else None,
            "path_len": self.path_len if self.found else None,
            "nodes_expanded": self.nodes_expanded,
            "max_frontier": self.max_frontier,
            "runtime_ms": round(self.runtime_s * 1000, 3),
        }


# --------------------------------------------------------------------------- #
# Heuristics
# --------------------------------------------------------------------------- #
def manhattan(a: Cell, b: Cell) -> int:
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def inadmissible(weight: float) -> Callable[[Cell, Cell], float]:
    """Manhattan distance inflated by `weight` (> 1 makes it inadmissible).

    Inflated heuristics expand fewer nodes but can lose optimality. Students
    use this to see the speed/optimality trade directly.
    """

    def h(a: Cell, b: Cell) -> float:
        return weight * manhattan(a, b)

    return h


# --------------------------------------------------------------------------- #
# Path reconstruction
# --------------------------------------------------------------------------- #
def _reconstruct(came_from: Dict[Cell, Optional[Cell]], goal: Cell) -> List[Cell]:
    path = [goal]
    node = goal
    while came_from.get(node) is not None:
        node = came_from[node]
        path.append(node)
    path.reverse()
    return path


def _path_cost(grid: Grid, path: List[Cell]) -> float:
    return sum(grid.entry_cost(cell) for cell in path[1:])


# --------------------------------------------------------------------------- #
# Uninformed search: BFS and DFS
# --------------------------------------------------------------------------- #
def bfs(grid: Grid) -> SearchResult:
    start, goal = grid.start, grid.goal
    t0 = time.perf_counter()
    frontier = deque([start])
    came_from: Dict[Cell, Optional[Cell]] = {start: None}
    order: List[Cell] = []
    expanded = 0
    max_frontier = 1
    while frontier:
        max_frontier = max(max_frontier, len(frontier))
        node = frontier.popleft()
        expanded += 1
        order.append(node)
        if node == goal:
            path = _reconstruct(came_from, goal)
            return SearchResult("BFS", True, path, _path_cost(grid, path),
                                expanded, max_frontier, time.perf_counter() - t0,
                                expanded_order=order)
        for nxt in grid.neighbors(node):
            if nxt not in came_from:
                came_from[nxt] = node
                frontier.append(nxt)
    return SearchResult("BFS", False, nodes_expanded=expanded,
                        max_frontier=max_frontier, runtime_s=time.perf_counter() - t0,
                        expanded_order=order)


def dfs(grid: Grid) -> SearchResult:
    start, goal = grid.start, grid.goal
    t0 = time.perf_counter()
    frontier = [start]
    came_from: Dict[Cell, Optional[Cell]] = {start: None}
    visited = set()
    order: List[Cell] = []
    expanded = 0
    max_frontier = 1
    while frontier:
        max_frontier = max(max_frontier, len(frontier))
        node = frontier.pop()
        if node in visited:
            continue
        visited.add(node)
        expanded += 1
        order.append(node)
        if node == goal:
            path = _reconstruct(came_from, goal)
            return SearchResult("DFS", True, path, _path_cost(grid, path),
                                expanded, max_frontier, time.perf_counter() - t0,
                                expanded_order=order)
        for nxt in grid.neighbors(node):
            if nxt not in visited:
                came_from.setdefault(nxt, node)
                frontier.append(nxt)
    return SearchResult("DFS", False, nodes_expanded=expanded, expanded_order=order,
                        max_frontier=max_frontier, runtime_s=time.perf_counter() - t0)


# --------------------------------------------------------------------------- #
# Best-first family: UCS, Greedy, A*, Weighted A*
# --------------------------------------------------------------------------- #
def _best_first(grid: Grid, label: str,
                g_weight: float, h_weight: float,
                heuristic: Callable[[Cell, Cell], float]) -> SearchResult:
    """Generic priority-queue search.

    Priority f = g_weight * g + h_weight * h. Special cases:
      UCS         : g_weight=1, h_weight=0
      Greedy      : g_weight=0, h_weight=1
      A*          : g_weight=1, h_weight=1
      Weighted A* : g_weight=1, h_weight=w (w > 1)
    """
    start, goal = grid.start, grid.goal
    t0 = time.perf_counter()
    g_score: Dict[Cell, float] = {start: 0.0}
    came_from: Dict[Cell, Optional[Cell]] = {start: None}
    counter = 0  # tie-breaker to keep heap stable and avoid comparing cells
    h0 = heuristic(start, goal)
    frontier: List[Tuple[float, int, Cell]] = [(g_weight * 0 + h_weight * h0, counter, start)]
    closed = set()
    order: List[Cell] = []
    expanded = 0
    max_frontier = 1
    while frontier:
        max_frontier = max(max_frontier, len(frontier))
        _, _, node = heapq.heappop(frontier)
        if node in closed:
            continue
        closed.add(node)
        expanded += 1
        order.append(node)
        if node == goal:
            path = _reconstruct(came_from, goal)
            return SearchResult(label, True, path, _path_cost(grid, path),
                                expanded, max_frontier, time.perf_counter() - t0,
                                expanded_order=order)
        for nxt in grid.neighbors(node):
            tentative = g_score[node] + grid.entry_cost(nxt)
            if tentative < g_score.get(nxt, float("inf")):
                g_score[nxt] = tentative
                came_from[nxt] = node
                counter += 1
                f = g_weight * tentative + h_weight * heuristic(nxt, goal)
                heapq.heappush(frontier, (f, counter, nxt))
    return SearchResult(label, False, nodes_expanded=expanded, expanded_order=order,
                        max_frontier=max_frontier, runtime_s=time.perf_counter() - t0)


def ucs(grid: Grid) -> SearchResult:
    return _best_first(grid, "UCS", 1.0, 0.0, manhattan)


def greedy(grid: Grid, heuristic: Callable[[Cell, Cell], float] = manhattan) -> SearchResult:
    return _best_first(grid, "Greedy", 0.0, 1.0, heuristic)


def astar(grid: Grid, heuristic: Callable[[Cell, Cell], float] = manhattan,
          label: str = "A*") -> SearchResult:
    return _best_first(grid, label, 1.0, 1.0, heuristic)


def weighted_astar(grid: Grid, weight: float = 2.0) -> SearchResult:
    return _best_first(grid, f"Weighted A* (w={weight})", 1.0, weight, manhattan)


# --------------------------------------------------------------------------- #
# Replanning scenario
# --------------------------------------------------------------------------- #
def replan_on_obstacle(grid: Grid, new_obstacle: Cell,
                       planner: Callable[[Grid], SearchResult] = astar):
    """Simulate a dynamic obstacle appearing on a planned path.

    Returns (initial, full_replan, partial_replan):
      initial        : the original plan
      full_replan    : replan from the start after the obstacle appears
      partial_replan : replan only from the cell just before the blockage

    The comparison shows that replanning from the agent's current position is
    much cheaper than replanning from scratch -- a core deployment point.
    """
    initial = planner(grid)
    if not initial.found or new_obstacle not in initial.path:
        return initial, None, None

    grid.block(new_obstacle)

    full_replan = planner(grid)

    idx = initial.path.index(new_obstacle)
    resume = initial.path[idx - 1] if idx > 0 else grid.start
    sub = Grid(grid.rows)  # rebuild to reset start, then re-apply obstacle
    sub.blocked = set(grid.blocked)
    sub.cost = dict(grid.cost)
    sub.start = resume
    sub.goal = grid.goal
    partial_replan = planner(sub)
    partial_replan.algorithm = "Partial replan"
    full_replan.algorithm = "Full replan"
    return initial, full_replan, partial_replan


# --------------------------------------------------------------------------- #
# Example maps
# --------------------------------------------------------------------------- #
MAPS: Dict[str, List[str]] = {
    # Mostly open: heuristic guidance pays off, BFS wastes effort.
    "open": [
        "S........",
        ".........",
        "....#....",
        "....#....",
        "....#....",
        ".........",
        "........G",
    ],
    # Maze: many dead ends; greedy gets misled, A* still efficient.
    "maze": [
        "S.#......",
        "..#.####.",
        "..#.#..#.",
        "....#.##.",
        "#####.#..",
        "....#.#.#",
        ".##.#.#..",
        ".#....#.G",
    ],
    # Non-uniform terrain: high-cost cells make the greedy/manhattan heuristic
    # misleading, so an inflated heuristic can produce a costlier path.
    "terrain": [
        "S99......",
        ".99......",
        ".99.####.",
        ".99......",
        "....99999",
        ".....111G",
    ],
    # Large open field: with a manhattan heuristic, plain A* expands a big
    # fraction of the grid (many cells tie on the optimal f-contour), while
    # Weighted A* drives straight at the goal and expands far fewer. The weight
    # sweep on this map shows a large, satisfying effort reduction.
    "open_large": [
        "S..............",
        "...............",
        "...............",
        "...............",
        "...............",
        "...............",
        "...............",
        "...............",
        "...............",
        "..............G",
    ],
}


def load_map(name: str, diagonal: bool = False) -> Grid:
    if name not in MAPS:
        raise KeyError(f"unknown map '{name}'; choices: {sorted(MAPS)}")
    return Grid(MAPS[name], diagonal=diagonal)


def render(grid: Grid, path: Optional[List[Cell]] = None) -> str:
    """ASCII render of a grid with an optional path overlaid as '*'."""
    path_set = set(path or [])
    lines = []
    for r in range(grid.height):
        chars = []
        for c in range(grid.width):
            cell = (r, c)
            if cell == grid.start:
                chars.append("S")
            elif cell == grid.goal:
                chars.append("G")
            elif cell in grid.blocked:
                chars.append("#")
            elif cell in path_set:
                chars.append("*")
            else:
                chars.append(grid.rows[r][c] if grid.rows[r][c].isdigit() else ".")
        lines.append("".join(chars))
    return "\n".join(lines)


def sweep_weight(grid_name: str,
                 weights=(1.0, 1.5, 2.0, 3.0, 5.0)) -> List[Tuple[float, SearchResult]]:
    """Run Weighted A* across a range of heuristic weights on one map.

    weight=1.0 is plain A* (optimal). As the weight grows the search expands
    fewer nodes but the path can get more expensive: students watch the
    speed/optimality trade emerge instead of being told about it.
    """
    return [(w, weighted_astar(load_map(grid_name), weight=w)) for w in weights]


def run_all(grid_name: str) -> List[SearchResult]:
    """Run the standard planner panel on a named map and return results."""
    results = []
    results.append(bfs(load_map(grid_name)))
    results.append(dfs(load_map(grid_name)))
    results.append(ucs(load_map(grid_name)))
    results.append(greedy(load_map(grid_name)))
    results.append(astar(load_map(grid_name)))
    results.append(astar(load_map(grid_name), inadmissible(3.0), label="A* (inflated h, w=3)"))
    results.append(weighted_astar(load_map(grid_name), weight=2.0))
    return results



print("search_lab loaded (embedded in this notebook -- no repo clone needed).")


In [ ]:
def show(results, title=None):
    """Print a panel of SearchResult objects as an aligned table."""
    if title:
        print(title)
    cols = ["algorithm", "found", "cost", "path_len", "nodes_expanded", "max_frontier", "runtime_ms"]
    rows = [r.as_row() for r in results]
    widths = {c: max(len(c), *(len(str(row[c])) for row in rows)) for c in cols}
    header = "  ".join(c.ljust(widths[c]) for c in cols)
    print(header); print("-" * len(header))
    for row in rows:
        print("  ".join(str(row[c]).ljust(widths[c]) for c in cols))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors


def _draw_grid_base(ax, grid):
    """Shade every cell: blocked (dark), terrain cost (light orange tint), free (light gray)."""
    ax.set_xlim(-0.5, grid.width - 0.5)
    ax.set_ylim(-0.5, grid.height - 0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.invert_yaxis()
    ax.set_aspect("equal")
    for r in range(grid.height):
        for c in range(grid.width):
            cell = (r, c)
            if cell in grid.blocked:
                color = "#2b2b2b"
            elif cell in grid.cost:
                t = min(grid.cost[cell] / 9.0, 1.0)
                color = (1.0, 1.0 - 0.55 * t, 1.0 - 0.85 * t)
            else:
                color = "#f4f6f8"
            ax.add_patch(mpatches.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                             facecolor=color, edgecolor="#d0d5db",
                                             linewidth=0.5, zorder=0))


def _mark_start_goal(ax, grid, start_size=90, goal_size=170):
    sr, sc = grid.start
    gr, gc = grid.goal
    ax.scatter([sc], [sr], marker="o", s=start_size, facecolor="#2ecc71",
               edgecolor="black", linewidths=1.0, zorder=4, label="start")
    ax.scatter([gc], [gr], marker="*", s=goal_size, facecolor="#e74c3c",
               edgecolor="black", linewidths=0.8, zorder=4, label="goal")


def plot_trajectory(grid, result, ax=None, title=None, cmap="viridis"):
    """Draw the gridworld with start/goal marked and the found path colored by
    step order (dark = early, bright = late) -- so the direction of travel is
    visible at a glance, not just an undifferentiated line."""
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(grid.width * 0.5 + 1.4, grid.height * 0.5 + 1))
    _draw_grid_base(ax, grid)

    if result.found and result.path:
        rows = [p[0] for p in result.path]
        cols = [p[1] for p in result.path]
        n = len(result.path)
        ax.plot(cols, rows, "-", color="#9aa5b1", linewidth=1.5, zorder=1, alpha=0.8)
        sc = ax.scatter(cols, rows, c=range(n), cmap=cmap, s=55, zorder=2,
                         edgecolors="white", linewidths=0.6)
        if own_fig:
            cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label("step index", fontsize=8)
            cbar.set_ticks([0, max(n - 1, 0)])
            cbar.set_ticklabels(["early", "late"])

    _mark_start_goal(ax, grid)

    found_str = "found" if result.found else "NOT FOUND"
    default_title = f"{result.algorithm} — {found_str}"
    if result.found:
        default_title += f" (cost={result.cost:.1f}, steps={result.path_len})"
    ax.set_title(title or default_title, fontsize=10)
    if own_fig:
        ax.legend(loc="upper left", fontsize=7, framealpha=0.9)
        fig.tight_layout()
        return fig


def plot_expansion_comparison(grid_name, results, cols=3, cmap_name="viridis"):
    """One panel per algorithm: every expanded cell colored by expansion order
    (dark = early, bright = late), with the final path traced on top -- so you
    can see *how* each search grew (BFS's ripple, DFS's snake, Greedy's
    beeline, A*'s cone), not just the final expanded-node count."""
    n = len(results)
    rows_n = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.2, rows_n * 2.9), squeeze=False)
    axes = axes.flatten()
    cmap = matplotlib.colormaps[cmap_name]

    for i, result in enumerate(results):
        ax = axes[i]
        grid = load_map(grid_name)
        _draw_grid_base(ax, grid)

        order = result.expanded_order
        if order:
            norm = mcolors.Normalize(vmin=0, vmax=max(len(order) - 1, 1))
            for k, (r, c) in enumerate(order):
                ax.add_patch(mpatches.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                                 facecolor=cmap(norm(k)), edgecolor="none",
                                                 zorder=1, alpha=0.9))

        if result.found and result.path:
            prows = [p[0] for p in result.path]
            pcols = [p[1] for p in result.path]
            ax.plot(pcols, prows, "-", color="white", linewidth=2.8, zorder=2)
            ax.plot(pcols, prows, "-", color="#1a1a2e", linewidth=1.2, zorder=3)

        _mark_start_goal(ax, grid, start_size=70, goal_size=140)
        ax.set_title(f"{result.algorithm}\nexpanded={result.nodes_expanded}", fontsize=9)

    for j in range(n, len(axes)):
        axes[j].axis("off")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.tolist(), fraction=0.02, pad=0.02, aspect=40)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(["early", "late"])
    cbar.set_label("expansion order", fontsize=8)

    fig.suptitle(f"Search expansion comparison — map: {grid_name}", fontsize=12)
    return fig

print("plot_trajectory and plot_expansion_comparison loaded.")


## Part A — Read the instrumentation on one map

`nodes_expanded` is the CPU proxy; `max_frontier` is the memory proxy. Watch how planners that find the **same path** can differ wildly in effort.

In [ ]:
show(run_all("maze"), title="=== maze ===")
print()
print(render(load_map("maze"), astar(load_map("maze")).path))

**Visualize it.** The table above tells you *how much* effort each planner spent; this plot shows *where* the winning path actually goes and in what order. Color runs from dark (near the start) to bright (near the goal) -- if the path visually backtracks, you'll see the colors jump instead of flowing smoothly.

In [ ]:
grid = load_map("maze")
result = astar(grid)
plot_trajectory(grid, result)
plt.show()


## Part B — Compare across map structures

- `open`: lots of free space and ties on the optimal f-contour.
- `maze`: dead ends and corridors.
- `terrain`: non-uniform step costs (digits are entry costs).

In [ ]:
for name in ("open", "maze", "terrain"):
    show(run_all(name), title=f"=== {name} ===")
    print()

**Look for:** a planner that finds a *valid but expensive* path (DFS on `terrain`), and a map where an admissible heuristic does NOT reduce search effort (ties on `open`).

**See the shape of the search.** Same six planners as the table above, now drawn as a heatmap of *when* each cell was expanded (dark = early, bright = late), with the final path traced on top. Compare `open` (lots of ties on the optimal f-contour -- watch BFS/UCS/A* balloon into a near-uniform ripple while Greedy and Weighted A* beeline straight at the goal) against `maze` (corridors force a much more similar expansion order across planners). Connect the shapes back to the `nodes_expanded` numbers you just read.

In [ ]:
for map_name in ("open", "maze"):
    results = [
        bfs(load_map(map_name)),
        dfs(load_map(map_name)),
        ucs(load_map(map_name)),
        greedy(load_map(map_name)),
        astar(load_map(map_name)),
        weighted_astar(load_map(map_name), weight=2.0),
    ]
    plot_expansion_comparison(map_name, results)
    plt.show()


## Part C — The heuristic effect

Same map, three heuristic regimes. An **inflated** heuristic expands fewer nodes; greedy ignores path cost entirely.

In [ ]:
name = "terrain"
panel = [
    ucs(load_map(name)),
    astar(load_map(name)),
    astar(load_map(name), inadmissible(3.0), label="A* inflated (w=3)"),
    greedy(load_map(name)),
    weighted_astar(load_map(name), weight=2.0),
]
show(panel, title=f"=== heuristic effect on {name} ===")

## Part D — Sweep the weight yourself (manipulate and discover)

Weighted A* uses `f = g + weight * h`. `weight = 1.0` is plain A*. **Change the weight and watch what happens** to search effort and to solution cost on a large open field.

In [ ]:
print('weight  nodes_expanded  cost')
for w, r in sweep_weight('open_large', weights=(1.0, 1.5, 2.0, 3.0, 5.0)):
    print(f'{w:>5}   {r.nodes_expanded:>13}   {r.cost:>4}')

**Discover it yourself:** what happened to `nodes_expanded` as the weight rose? What happened to `cost`? On this map Weighted A* kept the optimal cost even at weight 5 — *why might that be* (hint: this implementation re-opens nodes when it finds a cheaper route), and what guarantee does Weighted A* actually give about cost? Try the sweep on `terrain` and `maze` too.

## Part E — Replanning when the world changes

A dynamic obstacle appears on the planned path. Compare replanning **from scratch** vs **from the agent's current cell**.

In [ ]:
grid = load_map("open")
initial = astar(load_map("open"))
obstacle = initial.path[len(initial.path) // 2]
print(f"Dropping obstacle at {obstacle}\n")
init, full, partial = replan_on_obstacle(grid, obstacle, planner=astar)
show([init, full, partial], title="=== initial vs full replan vs partial replan ===")

## Worksheet (your deliverable)

### 1. Comparison table

Pick one map and fill from your runs:

| Planner | Found? | Cost | Nodes expanded | Max frontier | Runtime (ms) |
|---------|--------|------|----------------|--------------|--------------|
| BFS | | | | | |
| DFS | | | | | |
| UCS | | | | | |
| Greedy | | | | | |
| A* (admissible) | | | | | |
| A* (inflated) | | | | | |
| Weighted A* | | | | | |

### 2. Weight-sweep finding (from Part D)

| weight | nodes_expanded | cost |
|--------|----------------|------|
| 1.0 | | |
| 2.0 | | |
| 5.0 | | |

- In one sentence: what did inflating the weight buy you, and what did it (or did it not) cost?

### 3. Two findings (each tied to numbers)

- **Heuristic / search effort:** which choice reduced `nodes_expanded`, and did it cost optimality?
- **Operational weakness:** a planner that was *technically correct* but operationally unattractive, with numbers.

### 4. Planner choice under a constraint

- `Stated constraint:` (e.g. "replan within X ms", "frontier under N", "cost within 10% of optimal")
- `Planner you would deploy:`
- `Tradeoff you are accepting:`
- `Evidence (numbers) that justify it:`

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking